# 01｜构建 T+1→T+11 收益标签

标签候选为 `in_universe_t & can_enter_t1`，与图片 `feature_valid` 解耦。每日先做 1%/99% Winsorize，再以 `ddof=0` 做截面 Z-score。

## 步骤 1：定位流水线并设置覆盖参数

In [ ]:
from pathlib import Path
import os

CURRENT_DIR = Path.cwd().resolve()
VIT_DIR = CURRENT_DIR if (CURRENT_DIR / 'vit_stage03_labels.py').exists() else CURRENT_DIR / 'ViT'
LABEL_STAGE_PATH = VIT_DIR / 'vit_stage03_labels.py'
os.environ['VIT_LABEL_OVERWRITE'] = '0'
print({'ViT目录': str(VIT_DIR), '覆盖标签': False})

## 步骤 2：构建年度标签

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, str(LABEL_STAGE_PATH)], cwd=VIT_DIR, check=True)

## 步骤 3：核对覆盖率与标准化

In [ ]:
import json
import pandas as pd
from IPython.display import display

manifest = json.loads((VIT_DIR / 'data/labels_t10/labels_manifest.json').read_text(encoding='utf-8'))
quality = pd.read_parquet(VIT_DIR / 'data/quality/label_quality_by_date.parquet')
quality['year'] = quality['trade_date'].astype(str).str[:4]
display(pd.Series({'标签签名': manifest['signature'], '有效标签': manifest['label_rows']}, name='value').to_frame())
display(quality.groupby('year', as_index=False).agg(label_candidate_rows=('label_candidate_rows', 'sum'), label_valid_rows=('label_valid_rows', 'sum'), max_abs_target_mean=('target_mean', lambda x: x.abs().max()), max_target_std_error=('target_std_ddof0', lambda x: (x - 1).abs().max())))